# Load metrics files for each version of a given experiment, and summarize performance in various tables

In [1]:
import pandas as pd
from functools import reduce
import ast
import numpy as np
import pyvista as pv
import json
from pathlib import Path
from config import METRICS_DIR

In [12]:
# sometimes I saved single metrics (like LDDMM) as [value], so I need to have it as single value ...
def parse_value(x):
    # numpy array or list from LDDMM
    if isinstance(x, (np.ndarray, list, tuple)):
        return float(x[0])
    # scalar from chamfer or haussdorff
    return float(x)

# for styling tables
def highlight_best_style(val, mask):
    return 'background-color: orange' if mask else ''

def highlight_top3(val, rank):
    if rank == 1:
        return 'background-color: orange'
    elif rank == 2:
        return 'background-color: gray'
    elif rank == 3:
        return 'background-color: peru'  # bronze-ish color
    else:
        return ''

In [3]:
experiment_name = "RegLambdaAndAnneal"
exp_subdir = "training_sweeps/"

In [4]:
# for these experiments, names are just like {version}-{experiment_name}-{metric}-{opt}.parquet
# retrieve all the wanted files
dfs = []

for file_path in METRICS_DIR.glob(f"*-{experiment_name}*.parquet"):

    df = pd.read_parquet(file_path)

    # go fetch the specs file to add columns I need to differentiate versions
    version = file_path.stem.split("-")[0].split("_")[-1]    
    
    df["version"] = int(version)

    exp = exp_subdir + file_path.stem.split("-")[1]
    with open(f"experiments/{exp}/version_{version}/hparams.json") as f:
        specs = json.load(f)

    # this has to be changed manually for what is wanted ...
    code_reg = specs["code_reg_lambda"]
    anneal = specs["anneal_reg_loss"]
    
    df["lambda_reg"] = code_reg 
    df["anneal"] = anneal 
    
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

df_all["value"] = df_all["value"].apply(parse_value)

df_all

,version,patient,organ,metric,value,lambda_reg,anneal
0,8,AF059,epicardium,chamfer,3422.372148,0.000001,True
1,8,AF059,la_endo,chamfer,2963.741028,0.000001,True
2,8,AF059,ra_endo,chamfer,3172.894699,0.000001,True
3,8,LEU_NORM_F017,epicardium,chamfer,2532.928803,0.000001,True
4,8,LEU_NORM_F017,la_endo,chamfer,2668.053686,0.000001,True
...,...,...,...,...,...,...,...
295,5,AF037,la_endo,chamfer,3652.467622,0.000100,False
296,5,AF037,ra_endo,chamfer,1964.403686,0.000100,False
297,5,LEU_NORM_0717,epicardium,chamfer,3547.518299,0.000100,False
298,5,LEU_NORM_0717,la_endo,chamfer,2558.378063,0.000100,False


## Normalize metrics (computed at the original (micrometers) mesh scale )
Values of all the metrics are computed on the meshes at their original scale (coordinates in micrometers). Normalize them per-patient and per-organ, using some caracteristic scale of the original meshes, in this case, use bounding box diagonal

In [ ]:
from config import PATIENT_MESHES_DIR
patients = df_all["patient"].unique()
organs = df_all["organ"].unique()
rows = [] 
for patient in patients: # retrieve carachteristic scale per patient, per organ, from original mesh
    for organ in organs:
        mesh_orig = pv.read(PATIENT_MESHES_DIR / patient / f"{organ}-processed.vtp")
        scale = mesh_orig.field_data["scale-tooriginalrange"]
        mesh_orig.points *= scale
        bounds = mesh_orig.bounds  # (xmin, xmax, ymin, ymax, zmin, zmax)
        dx = bounds[1] - bounds[0]
        dy = bounds[3] - bounds[2]
        dz = bounds[5] - bounds[4]
        bbox_diagonal_micrometers = np.sqrt(dx**2 + dy**2 + dz**2)
        
        rows.append(
            {
                "patient": patient,
                "organ": organ,
                "ref_length_micrometers": bbox_diagonal_micrometers
            }
        )

df_ref_len = pd.DataFrame(rows)

df_all_with_ref_len = df_all.merge(df_ref_len, on=["patient", "organ"], how="left")

df_all["value_norm"] = df_all_with_ref_len["value"] / df_all_with_ref_len["ref_length_micrometers"]

df_all

,version,patient,organ,metric,value,lambda_reg,anneal,value_norm
0,8,AF059,epicardium,chamfer,3422.372148,0.000001,True,0.021519
1,8,AF059,la_endo,chamfer,2963.741028,0.000001,True,0.024417
2,8,AF059,ra_endo,chamfer,3172.894699,0.000001,True,0.027182
3,8,LEU_NORM_F017,epicardium,chamfer,2532.928803,0.000001,True,0.016293
4,8,LEU_NORM_F017,la_endo,chamfer,2668.053686,0.000001,True,0.023535
...,...,...,...,...,...,...,...,...
295,5,AF037,la_endo,chamfer,3652.467622,0.000100,False,0.025118
296,5,AF037,ra_endo,chamfer,1964.403686,0.000100,False,0.014301
297,5,LEU_NORM_0717,epicardium,chamfer,3547.518299,0.000100,False,0.021953
298,5,LEU_NORM_0717,la_endo,chamfer,2558.378063,0.000100,False,0.022608


In [ ]:
df_all = df_all.drop(columns="version") # don't need it really, I added specific columns to identify what changed in each version

Separate each metric (if there are multiple)

In [ ]:
df_all = df_all.query(" metric == 'chamfer' ").drop(columns="metric")

Compute mean and std over all patients for each version and for each organ

In [8]:
df = df_all.groupby(["organ", "lambda_reg", "anneal"])["value_norm"].agg( 
    mean="mean",
    std="std"
).reset_index()

### Style into a table

In [10]:
table = (
    df
    .pivot_table(
        index=["lambda_reg", "anneal"],   # can accept multi-index rows
        columns="organ",
        values=["mean", "std"],
        aggfunc="mean"
    )
    .sort_index()
    .swaplevel(axis=1)
    .sort_index(axis=1)
)

table

organ             epicardium             la_endo             ra_endo          
                        mean       std      mean       std      mean       std
lambda_reg anneal                                                             
0.000001   False    0.021265  0.005387  0.027729  0.004998  0.024379  0.004605
           True     0.021176  0.005341  0.027967  0.006896  0.024742  0.004251
0.000010   False    0.017083  0.002693  0.018367  0.002601  0.015055  0.001941
           True     0.016559  0.002875  0.018586  0.002060  0.014525  0.001961
0.000100   False    0.020910  0.003787  0.022250  0.003816  0.016664  0.002804
           True     0.020785  0.003421  0.020464  0.003736  0.016231  0.002457
0.001000   False    0.031543  0.005018  0.034657  0.005838  0.024011  0.004157
           True     0.031782  0.004107  0.031409  0.005002  0.024435  0.005461
0.010000   False    0.068225  0.010822  0.084092  0.009672  0.083750  0.009088
           True     0.070512  0.011880  0.076767  0.012645  0.090412  0.009934

In [11]:
table_combined = pd.DataFrame(index=table.index)

for organ in table.columns.levels[0]:
    mean_col = (organ, 'mean')
    std_col = (organ, 'std')
    
    # Format as "mean ± std"
    table_combined[organ] = (
        table[mean_col].round(4).astype(str) + " ± " + table[std_col].round(4).astype(str)
    )
table_combined

epicardium          la_endo          ra_endo
lambda_reg anneal                                                   
0.000001   False   0.0213 ± 0.0054   0.0277 ± 0.005  0.0244 ± 0.0046
           True    0.0212 ± 0.0053   0.028 ± 0.0069  0.0247 ± 0.0043
0.000010   False   0.0171 ± 0.0027  0.0184 ± 0.0026  0.0151 ± 0.0019
           True    0.0166 ± 0.0029  0.0186 ± 0.0021   0.0145 ± 0.002
0.000100   False   0.0209 ± 0.0038  0.0223 ± 0.0038  0.0167 ± 0.0028
           True    0.0208 ± 0.0034  0.0205 ± 0.0037  0.0162 ± 0.0025
0.001000   False    0.0315 ± 0.005  0.0347 ± 0.0058   0.024 ± 0.0042
           True    0.0318 ± 0.0041   0.0314 ± 0.005  0.0244 ± 0.0055
0.010000   False   0.0682 ± 0.0108  0.0841 ± 0.0097  0.0838 ± 0.0091
           True    0.0705 ± 0.0119  0.0768 ± 0.0126  0.0904 ± 0.0099

### highlight best values

In [13]:
# Extract only mean values for comparison
table_means = table.xs('mean', axis=1, level=1)
# Best lambda_reg per column (per organ)
best_mask = table_means.eq(table_means.min(axis=0), axis=1)
# Rank the values per column (ascending because lower is better)
ranks = table_means.rank(method='min', axis=0)  # smallest = rank 1

In [14]:
# Apply style
table_styled = table_combined.style.apply(
    lambda row: [highlight_best_style(v, b) for v, b in zip(row, best_mask.loc[row.name])],
    axis=1
)
table_styled

In [15]:
table_styled = table_combined.style.apply(
    lambda row: [
        highlight_top3(v, ranks.loc[row.name, col])
        for col, v in zip(table_means.columns, row)
    ],
    axis=1
)
table_styled